# Q&A agent

In [ ]:
import boto3
import json
import psycopg2
import psycopg2.extras
import os
import re
from requests_aws4auth import AWS4Auth
from opensearchpy import OpenSearch, RequestsHttpConnection

bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')

CLAUDE_MODEL  = 'us.anthropic.claude-haiku-4-5-20251001-v1:0'
TITAN_MODEL   = 'amazon.titan-embed-text-v2:0'
OS_ENDPOINT   = os.environ['OPENSEARCH_ENDPOINT']
OS_INDEX      = 'invoices'
REGION        = 'us-east-1'

DB_HOST     = 'findoc-relational-db.cmp0smys0m7u.us-east-1.rds.amazonaws.com'
DB_PORT     = 5432
DB_NAME     = 'postgres'
DB_USER     = 'postgres'
DB_PASSWORD = 'QDzMzO5U0nDT6D7Qyo79'

DB_SCHEMA = """
Tables available:
1. invoices (id, s3_key, vendor_name, vendor_address, invoice_number, invoice_date, due_date,
   po_number, subtotal, tax_amount, tax_rate, total_amount, currency, payment_terms,
   bank_details, notes, line_items JSONB, raw_json JSONB, created_at)
2. receipts (id, s3_key, merchant_name, merchant_address, receipt_number, receipt_date,
   subtotal, tax_amount, tax_rate, total_amount, currency, payment_method,
   items JSONB, notes, raw_json JSONB, created_at)
"""

def get_db_conn():
    return psycopg2.connect(
        host=DB_HOST, port=DB_PORT, dbname=DB_NAME,
        user=DB_USER, password=DB_PASSWORD,
        connect_timeout=10, sslmode='require'
    )

def get_os_client():
    session = boto3.Session()
    creds   = session.get_credentials().get_frozen_credentials()
    auth    = AWS4Auth(creds.access_key, creds.secret_key,
                       REGION, 'aoss', session_token=creds.token)
    return OpenSearch(
        hosts=[{'host': OS_ENDPOINT.replace('https://', ''), 'port': 443}],
        http_auth=auth, use_ssl=True,
        connection_class=RequestsHttpConnection
    )

def call_claude(system_prompt, user_message, max_tokens=1000):
    resp = bedrock.invoke_model(
        modelId=CLAUDE_MODEL,
        body=json.dumps({
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": max_tokens,
            "system": system_prompt,
            "messages": [{"role": "user", "content": user_message}]
        })
    )
    return json.loads(resp['body'].read())['content'][0]['text'].strip()

def classify_question(question):
    system = """You are a question classifier. Classify the user's question as either:
- "semantic": questions about finding, searching, describing documents, vendors, content
- "math": questions involving calculations, sums, totals, counts, averages, comparisons, aggregations

Respond with ONLY one word: semantic or math"""
    result = call_claude(system, question)
    return 'math' if 'math' in result.lower() else 'semantic'

def get_titan_embedding(text):
    resp = bedrock.invoke_model(
        modelId=TITAN_MODEL,
        body=json.dumps({"inputText": text[:8000]})
    )
    return json.loads(resp['body'].read())['embedding']

def semantic_search(question, top_k=5):
    os_client = get_os_client()
    query = {
    "size": top_k,
    "query": {
        "query_string": {
            "query": "*",
            "default_field": "*"
        }
    }
    }
    resp = os_client.search(index=OS_INDEX, body=query)
    hits = resp['hits']['hits']
    results = []
    for hit in hits:
        src = hit['_source']
        results.append({
            "vendor": src.get("vendor_name"),
            "invoice_number": src.get("invoice_number"),
            "date": src.get("invoice_date"),
            "total": src.get("total_amount"),
            "currency": src.get("currency"),
            "notes": src.get("notes"),
            "payment_terms": src.get("payment_terms"),
        })
    return results

def generate_sql(question):
    system = f"""You are a PostgreSQL expert. Generate a SQL query to answer the user's question.
{DB_SCHEMA}
Rules:
- Return ONLY the SQL query, no explanation, no markdown, no backticks
- Use proper PostgreSQL syntax
- For date operations use invoice_date or receipt_date columns
- tax_amount and total_amount are NUMERIC columns
- Always add LIMIT 100 unless the question asks for aggregates
- Never use DROP, DELETE, UPDATE, INSERT, ALTER or any destructive operations"""
    sql = call_claude(system, question)
    sql = sql.strip().strip('`').strip()
    if sql.lower().startswith('sql'):
        sql = sql[3:].strip()
    return sql

def run_sql(sql):
    conn = get_db_conn()
    try:
        with conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
            cur.execute(sql)
            rows = cur.fetchall()
            return [dict(row) for row in rows]
    finally:
        conn.close()

def format_answer(question, query_type, raw_results, sql=None):
    if query_type == 'math':
        context = f"SQL query used: {sql}\nResults: {json.dumps(raw_results, default=str)}"
    else:
        context = f"Relevant documents found: {json.dumps(raw_results, default=str)}"

    system = """You are a helpful financial document assistant.
Answer the user's question clearly and concisely based on the data provided.
Format numbers nicely. If results are empty, say so clearly."""

    user_msg = f"""Question: {question}

Data from database:
{context}

Please provide a clear, helpful answer."""

    return call_claude(system, user_msg, max_tokens=500)

def generate_chart(question, raw_results):
    """Ask Claude to decide the best chart type and structure the data for it."""
    system = """You are a data visualization expert. Given a question and query results,
decide the best chart type and return ONLY a valid JSON object with this exact structure:
{
  "type": "bar" | "pie" | "line" | "doughnut" | "none",
  "title": "Chart title",
  "x_label": "X axis label (for bar/line)",
  "y_label": "Y axis label (for bar/line)",
  "data": [
    {"label": "string", "value": number}
  ]
}

Chart selection rules:
- pie or doughnut: for proportions, distributions, breakdowns (e.g. % per vendor, per currency)
- bar: for comparisons between categories (e.g. totals per vendor)
- line: for trends over time (e.g. monthly totals)
- none: for single-value results (e.g. SUM that returns one number)

Return ONLY the JSON, no explanation, no markdown, no backticks."""

    user_msg = f"""Question: {question}
Data: {json.dumps(raw_results, default=str)}"""

    raw = call_claude(system, user_msg, max_tokens=800)
    try:
        raw = raw.strip().strip('`')
        if raw.lower().startswith('json'):
            raw = raw[4:].strip()
        return json.loads(raw)
    except:
        return {"type": "none"}

def handler(event, context):
    question = event.get('question', '')
    if not question:
        return {"error": "No question provided"}

    print(f"Question: {question}")

    q_type = classify_question(question)
    print(f"Type: {q_type}")

    if q_type == 'math':
        sql = generate_sql(question)
        print(f"SQL: {sql}")
        try:
            results = run_sql(sql)
            answer = format_answer(question, 'math', results, sql)
            chart = generate_chart(question, results)
        except Exception as e:
            return {"error": f"SQL execution failed: {str(e)}", "sql": sql}
    else:
        results = semantic_search(question)
        answer = format_answer(question, 'semantic', results)
        chart = {"type": "none"}

    return {
        "question": question,
        "type": q_type,
        "answer": answer,
        "chart": chart,
        "raw_results": results if q_type == 'math' else [r['vendor'] for r in results]
    }

# Vector DB

In [ ]:
import boto3, json, base64, os
from requests_aws4auth import AWS4Auth
from opensearchpy import OpenSearch, RequestsHttpConnection

s3      = boto3.client('s3')
bedrock = boto3.client('bedrock-runtime', region_name='us-east-1')

CLAUDE_MODEL = 'us.anthropic.claude-haiku-4-5-20251001-v1:0'
TITAN_MODEL  = 'amazon.titan-embed-text-v2:0'
OUT_BUCKET   = os.environ['OUT_BUCKET']
OS_ENDPOINT  = os.environ['OPENSEARCH_ENDPOINT']
OS_INDEX     = 'invoices'
REGION       = 'us-east-1'

PROMPT = """You are a financial document extraction expert.
Examine this document image carefully.

First, determine if this is a financial document (invoice, receipt, purchase order, bill).
If it is NOT a financial document, return exactly: {"doc_type": "invalid", "reason": "not a financial document"}

If it IS a financial document, extract ALL relevant fields and return ONLY a valid JSON object with:
vendor_name, vendor_address, invoice_number, invoice_date, due_date,
po_number, line_items (array), subtotal, tax_amount, tax_rate,
total_amount, currency, payment_terms, bank_details, notes,
doc_type ("invoice" or "receipt").
Use null for missing fields. Do not include any text outside the JSON."""

def validate_document(extracted):
    if extracted.get('doc_type') == 'invalid':
        return False, extracted.get('reason', 'not a financial document')

    has_amount  = extracted.get('total_amount') not in (None, 'null', '', 0)
    has_vendor  = extracted.get('vendor_name')  not in (None, 'null', '', 'None')
    has_date    = (extracted.get('invoice_date') or extracted.get('receipt_date')) not in (None, 'null', '')
    has_number  = (extracted.get('invoice_number') or extracted.get('receipt_number')) not in (None, 'null', '')
    has_items   = bool(extracted.get('line_items'))

    score = sum([has_amount, has_vendor, has_date, has_number, has_items])

    if score < 2:
        missing = []
        if not has_amount:  missing.append('total_amount')
        if not has_vendor:  missing.append('vendor_name')
        if not has_date:    missing.append('date')
        if not has_number:  missing.append('document_number')
        return False, f"insufficient financial fields (score {score}/5) — missing: {', '.join(missing)}"

    return True, 'ok'

def create_index():
    import requests

    session = boto3.Session()
    creds   = session.get_credentials().get_frozen_credentials()
    auth    = AWS4Auth(
        creds.access_key,
        creds.secret_key,
        REGION,
        'aoss',
        session_token=creds.token
    )

    url  = f"{OS_ENDPOINT}/{OS_INDEX}"
    body = {
        "mappings": {
            "dynamic": True,
            "properties": {
                "embedding": {"type": "float"}
            }
        }
    }

    resp = requests.put(
        url,
        auth=auth,
        json=body,
        headers={"Content-Type": "application/json"}
    )
    print(f"Status: {resp.status_code}")
    print(f"Response: {resp.text}")
    return {"status": resp.status_code, "body": resp.text}

def get_os_client():
    session = boto3.Session()
    creds   = session.get_credentials().get_frozen_credentials()
    auth    = AWS4Auth(creds.access_key, creds.secret_key,
                       REGION, 'aoss', session_token=creds.token)
    return OpenSearch(
        hosts=[{'host': OS_ENDPOINT.replace('https://',''), 'port': 443}],
        http_auth=auth, use_ssl=True,
        connection_class=RequestsHttpConnection
    )

def extract_with_claude(image_bytes, mime_type):
    img_b64 = base64.b64encode(image_bytes).decode()
    resp = bedrock.invoke_model(
        modelId=CLAUDE_MODEL,
        body=json.dumps({
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 2048,
            "messages": [{
                "role": "user",
                "content": [
                    {"type": "image", "source": {
                        "type": "base64",
                        "media_type": mime_type,
                        "data": img_b64
                    }},
                    {"type": "text", "text": PROMPT}
                ]
            }]
        })
    )
    raw = json.loads(resp['body'].read())['content'][0]['text']
    raw = raw.strip().removeprefix('```json').removesuffix('```').strip()
    return json.loads(raw)

def get_titan_embedding(text):
    resp = bedrock.invoke_model(
        modelId=TITAN_MODEL,
        body=json.dumps({"inputText": text[:8000]})
    )
    return json.loads(resp['body'].read())['embedding']

def log_rejected(bucket, key, reason):
    out_key = key.rsplit('.', 1)[0] + '_rejected.json'
    s3.put_object(
        Bucket=OUT_BUCKET,
        Key=f"rejected/{out_key}",
        Body=json.dumps({"s3_key": key, "reason": reason}, indent=2),
        ContentType='application/json'
    )
    print(f"REJECTED: {key} — {reason}")

def process_file(bucket, key):
    ext  = key.lower().split('.')[-1]
    mime = 'image/jpeg' if ext in ('jpg', 'jpeg') else 'image/png'
    obj  = s3.get_object(Bucket=bucket, Key=key)
    data = obj['Body'].read()

    extracted = extract_with_claude(data, mime)

    is_valid, reason = validate_document(extracted)
    if not is_valid:
        log_rejected(bucket, key, reason)
        return {"status": "rejected", "key": key, "reason": reason}

    doc_type = str(extracted.get('doc_type', 'invoice')).lower()
    if 'receipt' not in doc_type:
        doc_type = 'invoice'

    embed_text = ' '.join(filter(None, [
        str(extracted.get('vendor_name', '')    or ''),
        str(extracted.get('invoice_number', '') or ''),
        str(extracted.get('invoice_date', '')   or ''),
        str(extracted.get('total_amount', '')   or ''),
        str(extracted.get('currency', '')       or ''),
        str(extracted.get('notes', '')          or ''),
        str(extracted.get('payment_terms', '')  or ''),
        doc_type
    ]))
    vector = get_titan_embedding(embed_text)

    os_client = get_os_client()
    os_client.index(index=OS_INDEX, body={
        **extracted,
        "embedding": vector,
        "s3_key":    key,
        "s3_bucket": bucket,
        "doc_type":  doc_type
    })

    out_key = key.rsplit('.', 1)[0] + '.json'
    s3.put_object(
        Bucket=OUT_BUCKET,
        Key=f"extracted/{out_key}",
        Body=json.dumps(extracted, indent=2),
        ContentType='application/json'
    )

    print(f"OK [{doc_type}]: {key} -> {extracted.get('vendor_name')}")
    return {"status": "ok", "doc_type": doc_type, "vendor": extracted.get('vendor_name')}

def handler(event, context):
    if event.get('action') == 'create_index':
        return create_index()
    elif 'Records' in event:
        results = []
        for record in event['Records']:
            result = process_file(
                record['s3']['bucket']['name'],
                record['s3']['object']['key']
            )
            results.append(result)
        return results
    elif 'bucket' in event and 'key' in event:
        return process_file(event['bucket'], event['key'])

# prostgress

In [ ]:
import boto3
import json
import psycopg2
import os

s3 = boto3.client('s3')

DB_HOST = 'findoc-relational-db.cmp0smys0m7u.us-east-1.rds.amazonaws.com'
DB_PORT     = 5432
DB_NAME     = 'postgres'
DB_USER     = 'postgres'
DB_PASSWORD = 'QDzMzO5U0nDT6D7Qyo79'
IN_BUCKET   = 'findoc-processed-output'

def get_db_conn():
    return psycopg2.connect(
        host=DB_HOST,
        port=DB_PORT,
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        connect_timeout=10,
        sslmode='require'
    )

def setup_tables(conn):
    with conn.cursor() as cur:
        cur.execute("""
            CREATE TABLE IF NOT EXISTS invoices (
                id                SERIAL PRIMARY KEY,
                s3_key            TEXT UNIQUE,
                vendor_name       TEXT,
                vendor_address    TEXT,
                invoice_number    TEXT,
                invoice_date      DATE,
                due_date          DATE,
                po_number         TEXT,
                subtotal          NUMERIC(12,2),
                tax_amount        NUMERIC(12,2),
                tax_rate          NUMERIC(6,4),
                total_amount      NUMERIC(12,2),
                currency          TEXT,
                payment_terms     TEXT,
                bank_details      TEXT,
                notes             TEXT,
                line_items        JSONB,
                raw_json          JSONB,
                created_at        TIMESTAMP DEFAULT NOW()
            );
        """)
        cur.execute("""
            CREATE TABLE IF NOT EXISTS receipts (
                id                SERIAL PRIMARY KEY,
                s3_key            TEXT UNIQUE,
                merchant_name     TEXT,
                merchant_address  TEXT,
                receipt_number    TEXT,
                receipt_date      DATE,
                subtotal          NUMERIC(12,2),
                tax_amount        NUMERIC(12,2),
                tax_rate          NUMERIC(6,4),
                total_amount      NUMERIC(12,2),
                currency          TEXT,
                payment_method    TEXT,
                items             JSONB,
                notes             TEXT,
                raw_json          JSONB,
                created_at        TIMESTAMP DEFAULT NOW()
            );
        """)
        conn.commit()
    print("Tables ready.")

def safe_numeric(val):
    if val is None:
        return None
    try:
        return float(str(val).replace(',', '').replace('$', '').strip())
    except:
        return None

def safe_date(val):
    if not val:
        return None
    try:
        from datetime import datetime
        for fmt in ('%Y-%m-%d', '%d/%m/%Y', '%m/%d/%Y', '%d-%m-%Y', '%B %d, %Y', '%b %d, %Y'):
            try:
                return datetime.strptime(str(val).strip(), fmt).date()
            except:
                continue
        return None
    except:
        return None

def classify_doc(data):
    doc_type = str(data.get('doc_type', '')).lower()
    if 'receipt' in doc_type:
        return 'receipt'
    vendor = str(data.get('vendor_name', '') or '').lower()
    inv_num = str(data.get('invoice_number', '') or '').lower()
    if inv_num and inv_num != 'none' and inv_num != 'null':
        return 'invoice'
    if 'receipt' in str(data).lower()[:200]:
        return 'receipt'
    return 'invoice'

def insert_invoice(conn, s3_key, data):
    with conn.cursor() as cur:
        cur.execute("""
            INSERT INTO invoices (
                s3_key, vendor_name, vendor_address, invoice_number,
                invoice_date, due_date, po_number,
                subtotal, tax_amount, tax_rate, total_amount,
                currency, payment_terms, bank_details, notes,
                line_items, raw_json
            ) VALUES (
                %s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s
            )
            ON CONFLICT (s3_key) DO UPDATE SET
                vendor_name    = EXCLUDED.vendor_name,
                total_amount   = EXCLUDED.total_amount,
                invoice_date   = EXCLUDED.invoice_date,
                raw_json       = EXCLUDED.raw_json;
        """, (
            s3_key,
            data.get('vendor_name'),
            data.get('vendor_address'),
            data.get('invoice_number'),
            safe_date(data.get('invoice_date')),
            safe_date(data.get('due_date')),
            data.get('po_number'),
            safe_numeric(data.get('subtotal')),
            safe_numeric(data.get('tax_amount')),
            safe_numeric(data.get('tax_rate')),
            safe_numeric(data.get('total_amount')),
            data.get('currency'),
            data.get('payment_terms'),
            str(data.get('bank_details', '')) if data.get('bank_details') else None,
            data.get('notes'),
            json.dumps(data.get('line_items') or []),
            json.dumps(data)
        ))
        conn.commit()

def insert_receipt(conn, s3_key, data):
    with conn.cursor() as cur:
        cur.execute("""
            INSERT INTO receipts (
                s3_key, merchant_name, merchant_address, receipt_number,
                receipt_date, subtotal, tax_amount, tax_rate, total_amount,
                currency, payment_method, items, notes, raw_json
            ) VALUES (
                %s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s
            )
            ON CONFLICT (s3_key) DO UPDATE SET
                merchant_name  = EXCLUDED.merchant_name,
                total_amount   = EXCLUDED.total_amount,
                receipt_date   = EXCLUDED.receipt_date,
                raw_json       = EXCLUDED.raw_json;
        """, (
            s3_key,
            data.get('vendor_name') or data.get('merchant_name'),
            data.get('vendor_address') or data.get('merchant_address'),
            data.get('receipt_number') or data.get('invoice_number'),
            safe_date(data.get('receipt_date') or data.get('invoice_date')),
            safe_numeric(data.get('subtotal')),
            safe_numeric(data.get('tax_amount')),
            safe_numeric(data.get('tax_rate')),
            safe_numeric(data.get('total_amount')),
            data.get('currency'),
            data.get('payment_method') or data.get('payment_terms'),
            json.dumps(data.get('line_items') or data.get('items') or []),
            data.get('notes'),
            json.dumps(data)
        ))
        conn.commit()

def process_all_json_files(conn):
    paginator = s3.get_paginator('list_objects_v2')
    pages = paginator.paginate(Bucket=IN_BUCKET, Prefix='extracted/')

    total, inserted, skipped = 0, 0, 0

    for page in pages:
        for obj in page.get('Contents', []):
            key = obj['Key']
            if not key.endswith('.json'):
                continue
            total += 1
            try:
                body = s3.get_object(Bucket=IN_BUCKET, Key=key)['Body'].read()
                data = json.loads(body)
                doc_type = classify_doc(data)

                if doc_type == 'receipt':
                    insert_receipt(conn, key, data)
                else:
                    insert_invoice(conn, key, data)

                print(f"OK [{doc_type}]: {key}")
                inserted += 1
            except Exception as e:
                print(f"SKIP {key}: {e}")
                skipped += 1

    return {"total": total, "inserted": inserted, "skipped": skipped}

def handler(event, context):
    action = event.get('action', 'populate')

    conn = get_db_conn()
    try:
        if action == 'setup_tables':
            setup_tables(conn)
            return {"status": "tables created"}

        elif action == 'populate':
            setup_tables(conn)
            result = process_all_json_files(conn)
            print(f"Done: {result}")
            return result

    finally:
        conn.close()